# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step template for loading and exploring a Mass Spectrometry dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # The Croissant Dataset metadata instance

print(f"Dataset loaded: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id` identifiers.

We enumerate all available RecordSets, their fields and their corresponding Field/Column `@id`s. This information is necessary for subsequent data extraction and referencing fields correctly.

In [ ]:
# Explore all available record sets and their structure.
from pprint import pprint
record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record sets:")
record_set_ids = []
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']} (name: {rs.get('name', '<no name>')})")
    record_set_ids.append(rs['@id'])
    if 'fields' in rs:
        print('  Fields:')
        for f in rs['fields']:
            print(f"    - @id: {f['@id']}, name: {f.get('name', '<no name>')}, dataType: {f.get('dataType', '<no type>')}")
    if 'columns' in rs:
        print('  Columns:')
        for c in rs['columns']:
            print(f"    - @id: {c['@id']}, name: {c.get('name', '<no name>')}, dataType: {c.get('dataType', '<no type>')}")
if not record_sets:
    print('No record sets found in the Croissant metadata.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified in the overview step above.

In [ ]:
# Make a list of available record set ids
record_sets = dataset.record_sets
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:  # Load only if records exist
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for record set {record_set_id} with {len(df)} rows and columns: {df.columns.tolist()}")
    else:
        print(f"No records found for record set '{record_set_id}'")
        dataframes[record_set_id] = pd.DataFrame()

# Display the columns of the first nonempty DataFrame
example_rs = None
for rs_id, df in dataframes.items():
    if not df.empty:
        example_rs = rs_id
        break
if example_rs is not None:
    print(f"\nColumns for '{example_rs}':")
    print(dataframes[example_rs].columns.tolist())
    dataframes[example_rs].head()
else:
    print('No nonempty record sets to preview.')

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing such as filtering, normalizing numeric fields, and grouping data. Operations here may include removing outliers, transforming distributions, or grouping by categorical attributes.

We select a record set and numeric field by `@id` for the demonstration.

In [ ]:
# Please refer to lists printed above to select these IDs in your own workflow.
# Here, we demonstrate with the first nonempty record set and try to select a numeric field.
import numpy as np

record_set_id = example_rs  # Use previously extracted non-empty table

if record_set_id is not None:
    df = dataframes[record_set_id]
    # Try to infer a numeric field
    possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not possible_numeric_fields:
        # Try to coerce columns to numeric
        for col in df.columns:
            df[col + '_num'] = pd.to_numeric(df[col], errors='coerce')
        possible_numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if possible_numeric_fields:
        numeric_field = possible_numeric_fields[0]
        print(f"Selected numeric field: {numeric_field} (from DataFrame '{record_set_id}')")
        # Remove NaN values
        filtered_df = df[[numeric_field]].dropna()
        threshold = filtered_df[numeric_field].mean()  # Use mean as an example filter
        filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())
        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / (filtered_df[numeric_field].std() + 1e-8)
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        # Try to group by the first non-numeric field
        group_fields = [col for col in df.columns if col != numeric_field and not pd.api.types.is_numeric_dtype(df[col])]
        if group_fields:
            group_field = group_fields[0]
            print(f"\nGrouping by field: '{group_field}'")
            grouped_df = df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(numeric_field, ascending=False)
            print(grouped_df.head())
        else:
            print("No non-numeric/categorical field found for grouping.")
    else:
        print('No numeric fields found in the DataFrame.')
else:
    print('No record sets available for EDA.')

## 5. Visualization
Visualize data distributions or relationships using common plotting tools. Here, we demonstrate with basic histograms/scatter plots.

In [ ]:
# Visualization: Histogram and scatter of numeric fields
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_id is not None and possible_numeric_fields:
    fig, axs = plt.subplots(1, min(2, len(possible_numeric_fields)), figsize=(12,4))
    for i, field in enumerate(possible_numeric_fields[:2]):
        sns.histplot(df[field].dropna(), kde=True, ax=axs[i] if len(possible_numeric_fields)>1 else axs, bins=30)
        axs[i].set_title(f"Distribution of '{field}'")
    plt.tight_layout()
    plt.show()
    # Scatter between first two numeric fields if present
    if len(possible_numeric_fields) >= 2:
        plt.figure(figsize=(6,4))
        sns.scatterplot(data=df, x=possible_numeric_fields[0], y=possible_numeric_fields[1])
        plt.title(f"Scatter: {possible_numeric_fields[0]} vs {possible_numeric_fields[1]}")
        plt.show()
else:
    print('No numeric fields available for visualization.')

## 6. Conclusion
Summarize key findings and observations from the FAIR² mass spectrometry dataset exploration. For this demonstration, we've programmatically loaded, inspected, and visualized real data from a Croissant-compliant research resource, referencing all entities by their `@id`. This modular workflow serves as a reproducible template for working with new scientific datasets in Python.